# 🏦 FinSight: Behavioral Finance Synthetic Data Generator

**Role:** Data Science (Data Generation Pipeline)

Notebook ini bertujuan untuk membangkitkan data sintetis (dummy data) mutasi rekening perbankan level industri (*Core Banking CIF*). Pembangkitan data ini tidak dilakukan secara acak (random) biasa, melainkan dikendalikan secara ketat oleh kerangka eksperimen **ADEMP**, distribusi probabilitas, dan psikologi keuangan (*Behavioral Finance*).

Data ini akan digunakan untuk melatih algoritma **K-Medoids Clustering** (pemetaan 3 persona: *Spendthrift, Unconflicted, Tightwad*) dan **Multinomial Naive Bayes** (Klasifikasi NLP transaksi tidak terstruktur).



## ⚙️ Parameter Target Data
* **Jumlah Nasabah:** 500 User dengan demografi menjulur ke kanan (Lognormal): 40% Mahasiswa, 40% First Jobber, 20% Profesional.
* **Durasi Transaksi:** 90 Hari (Bulan 1 hingga Bulan 3).
* **Target Output:** `df_nasabah` (500 baris profil CIF) dan `df_transaksi` (Puluhan ribu baris mutasi harian).
* **Reproduksibilitas:** WAJIB menggunakan `np.random.seed(42)` dan `Faker.seed(42)`.

## 🧬 Landasan Teori & AI Rules Engine
AI Assistant di Colab ini **WAJIB** mematuhi kerangka berikut saat berinteraksi dengan kode:
1.  **Distribusi Frekuensi (Poisson):** Jumlah transaksi harian diatur parameter $\lambda$. Disetel tinggi untuk mensimulasikan kebiasaan Gen-Z (*Micro-Transactions*).
2.  **Sistem Rem Otomatis (Dynamic Throttling):** Frekuensi belanja diatur oleh rasio Sisa Saldo / Gaji. Dibagi 3 fase: Sultan (>40%), Panik (15-40%), dan Survival (<15%).
3.  **Cause-and-Effect Persona:** Mayoritas nasabah memiliki persona statis. Hanya 15% nasabah dengan *flag* `is_dynamic` yang personanya bisa berubah (Trauma Finansial) JIKA mereka terlalu lama terjebak di mode "Survival" bulan sebelumnya.
4.  **Peak Season & Windfall:** Terdapat *multiplier* belanja *Wants* di awal bulan dan tanggal kembar. Serta ada peluang 2% per hari mendapat "Pemasukan Dadakan" (durian runtuh).
5.  **Injeksi Noise ML:** Terdapat 3 jenis *noise*: Biaya admin acak (Rp 1rb - Rp 6.5rb), Anomali harga *markup* berbasis Distribusi Normal (2% peluang, multiplier 3x-8x), dan *Typo/Alay Case* pada catatan transfer P2P (15%) untuk ujian model NLP.

In [ ]:
display(df_nasabah[df_nasabah['id_user'] == 'USR-103'])

,id_user,nama_nasabah,tanggal_lahir,nama_ibu_kandung,segmen_demografi,gaji_bulanan,persona_dasar,is_dynamic,persona_bulan_1,persona_bulan_2,persona_bulan_3
499,USR-103,Karen Usada,2006-05-29,Titin Mangunsong,Mahasiswa,2100000,Unconflicted,1,Unconflicted,Tightwad,Unconflicted


## 🗄️ Skema Dataframe
**1. Tabel Nasabah CIF (`df_nasabah`)**
* `id_user` (Primary Key: USR-001)
* `nama_nasabah` (Faker id_ID)
* `tanggal_lahir` & `nama_ibu_kandung` (PII untuk verifikasi KYC)
* `segmen_demografi` (Mahasiswa / First Jobber / Profesional)
* `gaji_bulanan` (Pendapatan dinamis sesuai segmen)
* `persona_dasar` (Spendthrift / Unconflicted / Tightwad)
* `is_dynamic` (Binary 0/1: Flag untuk potensi perubahan persona)

**2. Tabel Transaksi (`df_transaksi`)**
* `id_transaksi` (TRX-10001)
* `id_user` (Foreign Key)
* `timestamp` (Terurut kronologis 90 hari)
* `tipe_mutasi` (Kredit / Debit)
* `deskripsi_mutasi` (Merchant / TRF KE NAMA FAKER)
* `catatan_mutasi` (Target NLP: Notes transfer dengan injeksi *typo*)
* `mcc` (Merchant Category Code 4-digit dari Master Dictionary)
* `nominal` (Normal Distribution: $\mu$ dan $\sigma$)
* `sisa_saldo` (Running Balance anti-minus)
* `kategori_besar` & `kategori_detail` (Ground Truth COICOP)
* `label_anomali` (Binary 0/1: Flag untuk jebakan harga *markup* normal)

---
*Note untuk AI Colab: Gunakan list comprehension/batch appending untuk optimasi RAM. Do not hallucinate missing logic.*

In [ ]:
import pandas as pd
import numpy as np
!pip install faker
from faker import Faker
import random
from datetime import datetime, timedelta

# Profil Nasabah

In [ ]:
!pip install faker
from faker import Faker # Added import for Faker
import random # Added import for random
import numpy as np # Added import for numpy

# Inisialisasi Faker dan Seed agar hasil bisa direproduksi (konsisten jika di-run ulang)
fake = Faker('id_ID')
Faker.seed(42)
random.seed(42)
np.random.seed(42)

print("⏳ TAHAP 1: Membangkitkan Data Nasabah (Demografi Realistis)...")

jumlah_nasabah = 500
data_nasabah = []

# Kita bangkitkan sesuai porsi 40-40-20
for i in range(jumlah_nasabah):
    id_user = f"USR-{str(i+1).zfill(3)}"
    nama = fake.name()
    tanggal_lahir = fake.date_of_birth(minimum_age=18, maximum_age=35).strftime('%Y-%m-%d')
    nama_ibu_kandung = fake.name_female()

    # 🧠 ALOKASI DEMOGRAFI & GAJI
    if i < 200: # 40% Mahasiswa (200 orang)
        segmen = 'Mahasiswa'
        pendapatan = random.randint(150, 350) * 10000 # Rp 1.5 Juta - Rp 3.5 Juta
    elif i < 400: # 40% First Jobber (200 orang)
        segmen = 'First Jobber'
        pendapatan = random.randint(450, 600) * 10000 # Rp 4.5 Juta - Rp 6.0 Juta
    else: # 20% Profesional (100 orang)
        segmen = 'Profesional'
        pendapatan = random.randint(800, 2000) * 10000 # Rp 8.0 Juta - Rp 20.0 Juta

    # 🧠 INISIALISASI PERSONA
    base_persona = random.choice(['Spendthrift', 'Unconflicted', 'Tightwad'])

    # Flag Cause-and-Effect: Hanya 15% nasabah yang punya potensi berubah personanya
    is_dynamic = 1 if random.random() < 0.15 else 0

    data_nasabah.append([
        id_user, nama, tanggal_lahir, nama_ibu_kandung, segmen, pendapatan,
        base_persona, is_dynamic,
        base_persona, base_persona, base_persona # Inisialisasi persona untuk bulan 1, 2, 3
    ])

# Konversi ke DataFrame
df_nasabah = pd.DataFrame(data_nasabah, columns=[
    'id_user', 'nama_nasabah', 'tanggal_lahir', 'nama_ibu_kandung',
    'segmen_demografi', 'gaji_bulanan', 'persona_dasar', 'is_dynamic',
    'persona_bulan_1', 'persona_bulan_2', 'persona_bulan_3' # Menambahkan kolom persona bulanan
])

# Mengacak urutan baris agar segmen tidak berurutan saat dilihat
df_nasabah = df_nasabah.sample(frac=1, random_state=42).reset_index(drop=True)

print("✅ BINGO! Tabel df_nasabah Selesai!\n")
display(df_nasabah.head(20))

print("\n📊 RANGKUMAN DEMOGRAFI:")
print(df_nasabah['segmen_demografi'].value_counts())
print("\n📊 RANGKUMAN PERSONA DINAMIS (1 = Bisa Berubah, 0 = Statis):")
print(df_nasabah['is_dynamic'].value_counts())

⏳ TAHAP 1: Membangkitkan Data Nasabah (Demografi Realistis)...
✅ BINGO! Tabel df_nasabah Selesai!



,id_user,nama_nasabah,tanggal_lahir,nama_ibu_kandung,segmen_demografi,gaji_bulanan,persona_dasar,is_dynamic,persona_bulan_1,persona_bulan_2,persona_bulan_3
0,USR-362,Sutan Perkasa Hakim,1992-11-17,Gina Permata,First Jobber,4750000,Unconflicted,0,Unconflicted,Unconflicted,Unconflicted
1,USR-074,"Bakiadi Suryono, M.TI.",2002-04-22,dr. Cornelia Prakasa,Mahasiswa,3210000,Unconflicted,0,Unconflicted,Unconflicted,Unconflicted
2,USR-375,Karen Mulyani,2006-04-01,Cinthia Halimah,First Jobber,5580000,Tightwad,0,Tightwad,Tightwad,Tightwad
3,USR-156,Sari Laksita,2006-01-22,Lasmanto Hidayat,Mahasiswa,2970000,Spendthrift,0,Spendthrift,Spendthrift,Spendthrift
4,USR-105,"drg. Jaka Fujiati, M.Farm",1995-03-10,"R.A. Yance Irawan, S.T.",Mahasiswa,1600000,Tightwad,1,Tightwad,Tightwad,Tightwad
5,USR-395,R.A. Qori Wibisono,2002-07-18,Tgk. Harjo Mandala,First Jobber,4690000,Spendthrift,0,Spendthrift,Spendthrift,Spendthrift
6,USR-378,Eva Napitupulu,1996-04-17,Qori Prasasta,First Jobber,5960000,Unconflicted,0,Unconflicted,Unconflicted,Unconflicted
7,USR-125,Salimah Hutapea,1993-03-08,Kusuma Hutagalung,Mahasiswa,2140000,Spendthrift,1,Spendthrift,Spendthrift,Spendthrift
8,USR-069,Salimah Maheswara,2003-04-25,Ophelia Saragih,Mahasiswa,1550000,Tightwad,0,Tightwad,Tightwad,Tightwad
9,USR-451,"KH. Hendra Halim, S.T.",2005-12-18,Nardi Mustofa,Profesional,14400000,Unconflicted,0,Unconflicted,Unconflicted,Unconflicted



📊 RANGKUMAN DEMOGRAFI:
segmen_demografi
First Jobber    200
Mahasiswa       200
Profesional     100
Name: count, dtype: int64

📊 RANGKUMAN PERSONA DINAMIS (1 = Bisa Berubah, 0 = Statis):
is_dynamic
0    421
1     79
Name: count, dtype: int64


In [ ]:
print("Jumlah Persona per Bulan 1:")
print(df_nasabah['persona_bulan_1'].value_counts())
print("\nJumlah Persona per Bulan 2:")
print(df_nasabah['persona_bulan_2'].value_counts())
print("\nJumlah Persona per Bulan 3:")
print(df_nasabah['persona_bulan_3'].value_counts())

Jumlah Persona per Bulan 1:
persona_bulan_1
Spendthrift     174
Unconflicted    167
Tightwad        159
Name: count, dtype: int64

Jumlah Persona per Bulan 2:
persona_bulan_2
Spendthrift     174
Unconflicted    167
Tightwad        159
Name: count, dtype: int64

Jumlah Persona per Bulan 3:
persona_bulan_3
Spendthrift     174
Unconflicted    167
Tightwad        159
Name: count, dtype: int64


In [ ]:
df_nasabah.to_csv('df_nasabah.csv', index=False)
print("df_nasabah final (termasuk persona bulanan) berhasil disimpan sebagai 'df_nasabah.csv'")

df_nasabah final (termasuk persona bulanan) berhasil disimpan sebagai 'df_nasabah.csv'


# 🛒 Tahap 2: Master Dictionary Merchant (Pusat Kendali Harga & Simulasi MCC)

## 📌 Konteks untuk AI Assistant & Tim Engineer
Bagian ini bertugas membangun `master_merchant`, sebuah kamus data (*nested dictionary*) di Python yang berfungsi ganda sebagai **Ground Truth COICOP** dan **Engine Penentu Harga & MCC**.

**💡 Kenapa menggunakan Dictionary? (Simulasi Core Banking)**
Di sistem perbankan nyata, transaksi dari mesin EDC atau QRIS dikategorikan secara otomatis oleh bank menggunakan sistem **MCC (Merchant Category Code)** 4-digit di *backend* mereka. Karena kita mensimulasikan arsitektur *industry-level*, `master_merchant` ini bertindak sebagai fondasi untuk **Hybrid Classification System**:

1. **Rule-Based Engine (Untuk Merchant Resmi):** Transaksi di *merchant* komersial akan langsung dikategorikan menggunakan sandi MCC yang tertanam di *dictionary* ini (akurasi 100%).
2. **NLP Target Engine (Untuk Transfer P2P):** Khusus transaksi "Transfer ke Rekening Teman" (MCC 4829), sistem tidak bisa tahu uang itu untuk *Needs* atau *Wants*. Di sinilah model Machine Learning (TF-IDF + Naive Bayes) akan bekerja menebak kolom `catatan_mutasi` yang sudah kita injeksi dengan *typo*.
3. **K-Medoids Engine:** Perbedaan nilai rata-rata harga antar *merchant* ($\mu$) menciptakan rentang variasi pengeluaran yang menjadi fitur utama untuk klasterisasi persona finansial.

## ⚙️ Aturan Pembuatan Dictionary (WAJIB DIPATUHI OLEH AI):

**1. Hierarki 3 Level (COICOP, 50/30/20, dan Transfer):**
   * **Level 1 (Kategori Besar):** Harus terbagi menjadi 3 pilar: `"Needs"`, `"Wants"`, dan `"Transfer"`.
   * **Level 2 (Kategori Detail):** Mengikuti 7 pilar gaya hidup Gen-Z/Milenial (Misal: "F&B dan Nongkrong", "Tagihan & Utilitas", "Transportasi", "Kesehatan & Perawatan Diri", "Transfer P2P").
   * **Level 3 (Nama Merchant):** Ditulis menggunakan format kapital ala mutasi bank sesungguhnya (Misal: `"QRIS STARBUCKS"`, `"PAYMENT PLN MOBILE"`, `"TRF KE REKENING TEMAN"`).

**2. Tiga Parameter Wajib di Level Merchant:**
   Setiap *merchant* WAJIB memiliki tiga parameter parameter tersembunyi:
   * `mu`: Rata-rata harga wajar di *merchant* tersebut (Nilai Lognormal).
   * `sigma`: Standar deviasi (rentang variasi harga wajar).
   * `mcc`: String 4-digit sandi Merchant Category Code riil (Misal: F&B = '5814', Minimarket = '5411', Transfer P2P = '4829').

**3. Realita Kelas Sosial (Lifestyle Variance):** Di dalam satu kategori *Needs* atau *Wants*, WAJIB disediakan *merchant* kelas atas dan kelas bawah agar mesin K-Medoids bisa memisahkan *Spendthrift* dan *Tightwad*.
   * *Contoh F&B:* Ada `"QRIS STARBUCKS"` ($\mu$: 75.000) dan `"QRIS ES TEH KAMPUS"` ($\mu$: 5.000).
   * *Contoh Fashion:* Ada `"QRIS ZARA INDONESIA"` ($\mu$: 800.000) dan `"TRF THRIFT SHOP IG"` ($\mu$: 120.000).

---
*Note untuk AI Assistant: Jaga struktur nested dictionary ini agar tetap clean. Pastikan nilai `mu` disesuaikan dengan realita inflasi dan harga pasar di Indonesia saat ini (2026).*

## 🗂️ Arsitektur Klasifikasi MCC (Merchant Category Code)

Dalam sistem FinSight, kami menggunakan standar 4 digit MCC internasional. Transaksi dibagi menjadi dua jalur klasifikasi:

**Jalur 1: Rule-Based (Otomatis berdasarkan MCC Bank)**

| Kategori Besar | Kategori Detail (COICOP) | Contoh MCC | Deskripsi MCC |
| :--- | :--- | :--- | :--- |
| **Needs** | Tagihan & Utilitas | `4900` | Utilities (Listrik, Air, Gas, Internet) |
| **Needs** | Transportasi | `4121`, `5541` | Taxicabs/Ride-Hailing, Service Stations (SPBU) |
| **Needs** | Groceries & Kebutuhan Pokok | `5411` | Grocery Stores, Supermarkets |
| **Needs** | Kesehatan & Perawatan Diri | `5912`, `8099` | Pharmacies, Medical Services/Clinics |
| **Wants** | F&B dan Nongkrong | `5812`, `5814` | Eating Places (Restoran), Fast Food |
| **Wants** | Hiburan & Langganan | `4899`, `7832`, `7999`| Cable/Streaming, Cinemas, Recreation Services |
| **Wants** | Belanja Online & Fashion | `5311`, `5651` | Department Stores, Family Clothing Stores |

<br>

**Jalur 2: NLP/Machine Learning Target (Transfer P2P & Ekstraksi UI)**

| Kategori Besar (Raw Data) | Target Distribusi (UI / K-Medoids) | Kode MCC | Penanganan Kategori & Target NLP |
| :--- | :--- | :--- | :--- |
| **Transfer** | *P2P Transfer / E-Wallet* | `4829` | **Area Blank Spot.** Model **NLP (TF-IDF + Naive Bayes)** membaca `catatan_mutasi` untuk mendistribusikan aliran dana. |
| | ➡️ Masuk ke **Needs** | - | NLP mendeteksi *keywords* kebutuhan pokok (contoh: *"uang kos"*, *"patungan listrik"*). |
| | ➡️ Masuk ke **Wants** | - | NLP mendeteksi *keywords* konsumtif (contoh: *"bayar gacoan"*, *"tiket bioskop"*). |
| | ➡️ Masuk ke **Tabungan & Investasi** | - | **(Non-COICOP).** NLP mendeteksi *keywords* alokasi aset (*"RDN"*, *"Bibit"*). Memenuhi pilar 20% di sistem 50/30/20. |
| | ➡️ **Uncategorized** | - | Jika catatan kosong atau probabilitas NLP rendah. Dikecualikan dari perhitungan Persona K-Medoids. |

## Running Syntax

In [ ]:
print("⏳ TAHAP 2: Membangun Katalog Master Merchant (Cleaned)...")

master_merchant = {
    'Needs': {
        'Tagihan & Utilitas': {
            'PLN MOBILE': {'mu': 200000, 'sigma': 50000, 'mcc': '4900'},
            'PDAM': {'mu': 100000, 'sigma': 25000, 'mcc': '4900'},
            'INDIHOME': {'mu': 350000, 'sigma': 20000, 'mcc': '4900'},
            'UKT KAMPUS': {'mu': 5000000, 'sigma': 1000000, 'mcc': '8220'},
            'KPR BANK': {'mu': 3000000, 'sigma': 750000, 'mcc': '6012'},
            'PAKET DATA': {'mu': 50000, 'sigma': 25000, 'mcc': '4814'},
            'BIAYA KOSAN BULANAN': {'mu': 1200000, 'sigma': 300000, 'mcc': '6513'},
            'LAUNDRY KILOAN': {'mu': 35000, 'sigma': 10000, 'mcc': '7210'},
            'ICLOUD/GOOGLE ONE': {'mu': 45000, 'sigma': 0, 'mcc': '5734'}
        },
        'Transportasi': {
            'SPBU PERTAMINA': {'mu': 100000, 'sigma': 50000, 'mcc': '5541'},
            'SHELL INDONESIA': {'mu': 200000, 'sigma': 40000, 'mcc': '5541'},
            'GOJEK INDONESIA': {'mu': 35000, 'sigma': 15000, 'mcc': '4121'},
            'GRAB INDONESIA': {'mu': 40000, 'sigma': 15000, 'mcc': '4121'},
            'PARKIR': {'mu': 8000, 'sigma': 4000, 'mcc': '4121'},
            'CICILAN MOBIL': {'mu': 2500000, 'sigma': 500000, 'mcc': '6012'},
            'BENGKEL MOTOR': {'mu': 150000, 'sigma': 50000, 'mcc': '7538'},
            'KMT KRL COMMUTER': {'mu': 50000, 'sigma': 20000, 'mcc': '4111'},
            'E-TOLL FLAZZ': {'mu': 100000, 'sigma': 50000, 'mcc': '4784'}
        },
        'Groceries & Kebutuhan Pokok': {
            'INDOMARET': {'mu': 60000, 'sigma': 25000, 'mcc': '5411'},
            'ALFAMART': {'mu': 65000, 'sigma': 30000, 'mcc': '5411'},
            'SUPERINDO': {'mu': 300000, 'sigma': 120000, 'mcc': '5411'},
            'FOTOKOPI & PRINT': {'mu': 5000, 'sigma': 2000, 'mcc': '5111'},
            'WATSONS/GUARDIAN': {'mu': 100000, 'sigma': 50000, 'mcc': '5977'},
            'GALON & GAS': {'mu': 45000, 'sigma': 10000, 'mcc': '5499'}
        },
        'Kesehatan & Perawatan Diri': {
            'BPJS KESEHATAN': {'mu': 150000, 'sigma': 0, 'mcc': '8099'},
            'APOTEK KIMIA FARMA': {'mu': 85000, 'sigma': 30000, 'mcc': '5912'},
            'KLINIK PRATAMA': {'mu': 120000, 'sigma': 40000, 'mcc': '8099'},
            'ASURANSI JIWA': {'mu': 500000, 'sigma': 150000, 'mcc': '6300'},
            'HALODOC/ALODOKTER': {'mu': 60000, 'sigma': 20000, 'mcc': '8099'},
            'FITHUB/GYM': {'mu': 250000, 'sigma': 50000, 'mcc': '7997'}
        }
    },
    'Wants': {
        'F&B dan Nongkrong': {
            'STARBUCKS': {'mu': 75000, 'sigma': 20000, 'mcc': '5814'},
            'KOPI KENANGAN': {'mu': 35000, 'sigma': 10000, 'mcc': '5814'},
            'WARTEG': {'mu': 20000, 'sigma': 5000, 'mcc': '5812'},
            'SUSHI TEI': {'mu': 250000, 'sigma': 80000, 'mcc': '5812'},
            'ES TEH KAMPUS': {'mu': 5000, 'sigma': 1000, 'mcc': '5814'},
            'JAJANAN GEROBAK': {'mu': 10000, 'sigma': 3000, 'mcc': '5814'},
            'MIE GACOAN': {'mu': 30000, 'sigma': 15000, 'mcc': '5814'},
            'MIXUE INDONESIA': {'mu': 20000, 'sigma': 8000, 'mcc': '5814'},
            'RUMAH MAKAN PADANG': {'mu': 30000, 'sigma': 10000, 'mcc': '5812'},
            'GYU-KAKU AYCE': {'mu': 350000, 'sigma': 50000, 'mcc': '5812'},
            'CAFE AESTHETIC': {'mu': 65000, 'sigma': 20000, 'mcc': '5814'},
            'TEH TARIK PINGGIR JALAN': {'mu': 12000, 'sigma': 4000, 'mcc': '5814'},
            'NASI KUNING BU NUNUNG': {'mu': 18000, 'sigma': 5000, 'mcc': '5812'}
        },
        'Hiburan & Langganan': {
            'NETFLIX': {'mu': 186000, 'sigma': 0, 'mcc': '4899'},
            'SPOTIFY': {'mu': 54900, 'sigma': 0, 'mcc': '4899'},
            'CGV CINEMAS': {'mu': 60000, 'sigma': 15000, 'mcc': '7832'},
            'TIMEZONE': {'mu': 150000, 'sigma': 50000, 'mcc': '7999'},
            'YOUTUBE PREMIUM': {'mu': 59000, 'sigma': 0, 'mcc': '4899'},
            'TIX.ID / KONSER': {'mu': 150000, 'sigma': 50000, 'mcc': '7832'},
            'STEAM/VALORANT': {'mu': 120000, 'sigma': 50000, 'mcc': '7999'}
        },
        'Belanja Online & Fashion': {
            'SHOPEE': {'mu': 120000, 'sigma': 60000, 'mcc': '5311'},
            'TOKOPEDIA': {'mu': 150000, 'sigma': 80000, 'mcc': '5311'},
            'ZARA INDONESIA': {'mu': 500000, 'sigma': 250000, 'mcc': '5651'},
            'UNIQLO INDONESIA': {'mu': 450000, 'sigma': 150000, 'mcc': '5651'},
            'SOCIOLLA': {'mu': 250000, 'sigma': 80000, 'mcc': '5977'},
            'THRIFT SHOP IG/TIKTOK': {'mu': 100000, 'sigma': 40000, 'mcc': '5651'},
            'AKSESORIS HP ONLINE': {'mu': 30000, 'sigma': 15000, 'mcc': '5311'},
            'STICKER TELEGRAM/WA': {'mu': 10000, 'sigma': 0, 'mcc': '5734'}
        },
        'Produktivitas & Digital': {
            'CANVA PRO': {'mu': 50000, 'sigma': 0, 'mcc': '5734'},
            'NOTION': {'mu': 70000, 'sigma': 0, 'mcc': '5734'},
            'E-READER/WEBTOON': {'mu': 25000, 'sigma': 10000, 'mcc': '5734'},
            'FOTO STOCK ONLINE': {'mu': 15000, 'sigma': 5000, 'mcc': '5734'}
        }
    },
    'Transfer': {
        'Transfer P2P': {
            'REKENING TEMAN': {'mu': 50000, 'sigma': 30000, 'mcc': '4829'},
            'OVO/DANA/GOPAY': {'mu': 100000, 'sigma': 50000, 'mcc': '6540'}
        },
        'Investasi & Finansial': {
            'PT INDO PREMIER SEKURITAS': {'mu': 1000000, 'sigma': 300000, 'mcc': '6211'},
            'INDODAX': {'mu': 500000, 'sigma': 200000, 'mcc': '6211'},
            'BIBIT REKSADANA': {'mu': 300000, 'sigma': 150000, 'mcc': '6211'},
            'TOKOCRYPTO': {'mu': 250000, 'sigma': 100000, 'mcc': '6211'},
            'TABUNGAN EMAS PEGADAIAN': {'mu': 200000, 'sigma': 50000, 'mcc': '6012'},
            'MIRAE ASSET': {'mu': 1500000, 'sigma': 500000, 'mcc': '6211'},
            'STOCKBIT': {'mu': 1200000, 'sigma': 400000, 'mcc': '6211'},
            'AJAIB SEKURITAS': {'mu': 800000, 'sigma': 300000, 'mcc': '6211'},
            'REKSADANA BAREKSA': {'mu': 200000, 'sigma': 80000, 'mcc': '6211'},
            'KRIPTO PLUANG': {'mu': 300000, 'sigma': 120000, 'mcc': '6211'},
            'ORI/SUKUK RITEL': {'mu': 5000000, 'sigma': 1000000, 'mcc': '6211'},
            'EMAS DIGITAL INDOGOLD': {'mu': 150000, 'sigma': 60000, 'mcc': '6012'}
        }
    }
}

print("✅ BINGO! Master Dictionary Merchant Siap Digunakan!")

⏳ TAHAP 2: Membangun Katalog Master Merchant (Cleaned)...
✅ BINGO! Master Dictionary Merchant Siap Digunakan!


# ⚙️ Tahap 3: Mesin Waktu & Generator Transaksi (Looping 90 Hari)

## 📌 Konteks untuk AI Assistant & Tim Engineer
Bagian ini adalah *core engine* (mesin utama) yang akan merakit **Tabel Transaksi (`df_transaksi`)**. Mesin ini akan melakukan *looping* waktu selama 90 hari dan membangkitkan aktivitas finansial untuk ke-150 nasabah setiap harinya.

Generator ini **BUKAN** sekadar *randomizer* biasa. Mesin beroperasi berdasarkan prinsip **Behavioral Finance** dan simulasi ekosistem ekonomi nyata yang kompleks.

## 🧠 Aturan Logika Mesin (WAJIB DIPATUHI OLEH AI):

**1. Era Micro-Transaction (Poisson Distribution):**
Frekuensi jajan nasabah dalam sehari ditentukan oleh `np.random.poisson()`. Parameter dasar $\lambda$ diatur tinggi untuk mensimulasikan kebiasaan Gen-Z:
* **Spendthrift:** Needs ($\lambda=3.0$), Wants ($\lambda=5.0$), Transfer ($\lambda=1.5$).
* **Unconflicted:** Needs ($\lambda=3.0$), Wants ($\lambda=2.0$), Transfer ($\lambda=1.0$).
* **Tightwad:** Needs ($\lambda=2.5$), Wants ($\lambda=0.5$), Transfer ($\lambda=0.2$).

**2. Dynamic Throttling (Rem Otomatis Anti-Bangkrut):**
Frekuensi Poisson di atas BUKAN angka statis. Mesin wajib mengecek rasio `sisa_saldo / gaji_bulanan` setiap hari:
* **Sultan Mode (Saldo > 40%):** Jajan normal sesuai $\lambda$ dasar.
* **Panik Mode (Saldo 15% - 40%):** Frekuensi jajan *Wants* dan *Transfer* dipotong 50%.
* **Survival Mode (Saldo < 15%):** Frekuensi *Wants* dan *Transfer* DIBLOKIR (0). *Needs* dipotong hingga 30%. Jika nasabah terjebak di mode ini $\ge$ 7 hari berturut-turut, sistem wajib mencatatnya.

**3. Peak Season & Windfall Income:**
* **Gajian:** Tiap tanggal 1, masukkan `Kredit` sebesar `gaji_bulanan` (bukan 5 juta *flat*).
* **Tanggal Kalap:** Tambahkan *multiplier* pada $\lambda$ Wants jika berada di Post-Payday (Tgl 1-5, *multiplier* 1.5) atau Tanggal Kembar (misal 2.2, *multiplier* 2.5).
* **Windfall:** Peluang 2% per hari mendapat "Pemasukan Dadakan" (durian runtuh, profit trading).

**4. Cause-and-Effect Persona (Trauma Finansial):**
Tiap awal bulan baru, JIKA nasabah memiliki *flag* `is_dynamic == 1` DAN akumulasi "Survival Mode" mereka bulan lalu $\ge$ 7 hari, maka personanya diturunkan (Misal: *Spendthrift* $\rightarrow$ *Unconflicted*).

**5. Injeksi Noise Realita:**
* **Biaya Admin:** Peluang 30% harga ditambah *fee* acak (Rp 1rb - Rp 6.5rb).
* **Price Anomaly:** Peluang 2% harga di-*markup* menggunakan Multiplier acak antara 2x hingga 3x lipat dari harga wajar barang tersebut. Kolom `label_anomali` diisi 1.
* **Typo NLP Target:** Transaksi P2P (MCC 4829) wajib punya `catatan_mutasi` (*ground truth* Wants/Needs). Ada peluang 15% teks catatan tersebut disuntikkan *typo/alay case* sebagai ujian model Naive Bayes.

## Runing Syntax

In [ ]:
print("🚀 TAHAP 3: Menyalakan Mesin Waktu (Enhanced with Routines & Subs)...")

# --- PERSIAPAN VARIABEL ---
tanggal_awal = datetime(2026, 1, 1)
total_hari = 90
transaksi_list = []
id_trx_counter = 10001

list_windfall = ['PROFIT TRADING', 'HADIAH/GIFT', 'REFUND BELANJA', 'SIDE HUSTLE PROJECT', 'BONUS APRESIASI']
# --- DICTIONARY GROUND TRUTH P2P ---
# Format: "catatan_asli": ("Kategori_Besar", "Kategori_Detail")
dict_p2p_ground_truth = {
    # Kebutuhan Pokok (Needs)
    "uang kos": ("Needs", "Tagihan & Utilitas"),
    "patungan listrik": ("Needs", "Tagihan & Utilitas"),
    "patungan air": ("Needs", "Tagihan & Utilitas"),
    "beli token": ("Needs", "Tagihan & Utilitas"),
    "kas kelas": ("Needs", "Tagihan & Utilitas"),
    "fotokopi modul": ("Needs", "Groceries & Kebutuhan Pokok"),
    "bayar buku": ("Needs", "Groceries & Kebutuhan Pokok"),

    # Konsumtif (Wants)
    "bayar gacoan": ("Wants", "F&B dan Nongkrong"),
    "kopi tadi": ("Wants", "F&B dan Nongkrong"),
    "jajan gofood": ("Wants", "F&B dan Nongkrong"),
    "netflix": ("Wants", "Hiburan & Langganan"),
    "tiket bioskop": ("Wants", "Hiburan & Langganan"),
    "patungan spotify": ("Wants", "Hiburan & Langganan"),
    "utang mabar": ("Wants", "Hiburan & Langganan"),
    "hadiah": ("Wants", "Belanja Online & Fashion"),
    "beli baju": ("Wants", "Belanja Online & Fashion"),
    "sepatu thrifting": ("Wants", "Belanja Online & Fashion"),
    "kado ultah": ("Wants", "Belanja Online & Fashion")
}

poisson_lambda = {
    'Spendthrift': {'Needs': 3.0, 'Wants': 5.0, 'Transfer': 1.5},
    'Unconflicted': {'Needs': 3.0, 'Wants': 2.0, 'Transfer': 1.0},
    'Tightwad': {'Needs': 2.5, 'Wants': 0.5, 'Transfer': 0.2}
}

sisa_saldo_user = {row['id_user']: 0 for _, row in df_nasabah.iterrows()}
survival_streak = {row['id_user']: 0 for _, row in df_nasabah.iterrows()}
active_persona_user = {row['id_user']: row['persona_dasar'] for _, row in df_nasabah.iterrows()}

def inject_typo(text):
    if np.random.rand() < 0.15:
        noise_type = random.choice(['drop_char', 'alay_case', 'no_space'])
        if noise_type == 'drop_char' and len(text) > 4:
            idx = random.randint(0, len(text)-1)
            return text[:idx] + text[idx+1:]
        elif noise_type == 'alay_case':
            return "".join([c.upper() if random.random() > 0.5 else c.lower() for c in text])
        elif noise_type == 'no_space':
            return text.replace(" ", "")
    return text

# ==========================================
# ENGINE UTAMA
# ==========================================

# 1. FUNGSI GENERATOR TRANSAKSI
def process_cat(user_id, cat_name, count, current_date):
    global id_trx_counter
    for _ in range(count):
        dtl = random.choice(list(master_merchant[cat_name].keys()))
        mrc = random.choice(list(master_merchant[cat_name][dtl].keys()))
        pm = master_merchant[cat_name][dtl][mrc]

        price = abs(np.random.normal(pm['mu'], pm['sigma']))
        if np.random.rand() < 0.3: price += random.randint(1000, 6500)
        anom = 1 if np.random.rand() < 0.02 else 0
        if anom: price *= random.uniform(3.0, 8.0)

        if sisa_saldo_user[user_id] > price + 5000:
            sisa_saldo_user[user_id] -= price
            note = "-"
            m_final = mrc

            # 👇 SOLUSI BUG: Berikan nilai default yang sama dengan sistem bank
            gt_kategori_besar = cat_name
            gt_kategori_detail = dtl

            # --- LOGIKA P2P YANG BARU (Menimpa Ground Truth Default) ---
            if pm['mcc'] == '4829' and dtl != 'Investasi & Finansial':
                m_final = f"TRF KE {fake.name().upper()}"

                # Ambil satu catatan secara acak dari dictionary ground truth
                kata_asli = random.choice(list(dict_p2p_ground_truth.keys()))

                # Ekstrak Kunci Jawabannya (Menimpa nilai default di atas)
                gt_kategori_besar = dict_p2p_ground_truth[kata_asli][0]
                gt_kategori_detail = dict_p2p_ground_truth[kata_asli][1]

                # Rusak catatannya dengan typo untuk ujian NLP
                note = inject_typo(kata_asli)

            transaksi_list.append([
                f"TRX-{id_trx_counter}", user_id, current_date.replace(hour=random.randint(8,22)),
                'Debit', m_final, note, pm['mcc'], round(price, 2),
                round(sisa_saldo_user[user_id], 2), cat_name, dtl, anom,
                gt_kategori_besar, gt_kategori_detail # <--- SUDAH AMAN!
            ])
            id_trx_counter += 1

# 2. LOOPING WAKTU (90 HARI)
for hari in range(total_hari):
    current_date = tanggal_awal + timedelta(days=hari)
    is_weekend = current_date.weekday() >= 5

    peak_wants_multiplier = 1.0
    if current_date.day <= 5: peak_wants_multiplier = 1.5
    elif current_date.day == current_date.month: peak_wants_multiplier = 2.5
    if is_weekend: peak_wants_multiplier *= 1.3

    if current_date.day == 1:
        for index, row in df_nasabah.iterrows():
            user_id = row['id_user']
            gaji = row['gaji_bulanan']
            sisa_saldo_user[user_id] += gaji
            transaksi_list.append([
                f"TRX-{id_trx_counter}", user_id, current_date.replace(hour=7), 'Kredit',
                'TRF MASUK - PENDAPATAN TETAP', "-", '0000', float(gaji),
                round(sisa_saldo_user[user_id], 2), 'Pemasukan', 'Pendapatan Bulanan', 0
            ])
            id_trx_counter += 1

            # --- PERUBAHAN DISINI: Disesuaikan dengan nama baru di dictionary ---
            subs = [('NETFLIX', 'Wants', 'Hiburan & Langganan'), ('BPJS KESEHATAN', 'Needs', 'Kesehatan & Perawatan Diri')]
            for merchant_name, cat_big, cat_dtl in subs:
                p = master_merchant[cat_big][cat_dtl][merchant_name]
                cost = p['mu']
                if sisa_saldo_user[user_id] > cost:
                    sisa_saldo_user[user_id] -= cost
                    transaksi_list.append([
                        f"TRX-{id_trx_counter}", user_id, current_date.replace(hour=9), 'Debit',
                        merchant_name, "Auto-Debit Monthly", p['mcc'], float(cost),
                        round(sisa_saldo_user[user_id], 2), cat_big, cat_dtl, 0
                    ])
                    id_trx_counter += 1

            if hari > 0 and row['is_dynamic'] == 1 and survival_streak[user_id] >= 7:
                cp = active_persona_user[user_id]
                if cp == 'Spendthrift': active_persona_user[user_id] = 'Unconflicted'
                elif cp == 'Unconflicted': active_persona_user[user_id] = 'Tightwad'
            survival_streak[user_id] = 0
            df_nasabah.loc[df_nasabah['id_user'] == user_id, f'persona_bulan_{(hari//30)+1}'] = active_persona_user[user_id]

    for index, row in df_nasabah.iterrows():
        user_id = row['id_user']
        gaji = row['gaji_bulanan']

        if np.random.rand() < 0.02:
            win_val = round(abs(np.random.normal(500000, 300000)), -3)
            sisa_saldo_user[user_id] += win_val
            transaksi_list.append([
                f"TRX-{id_trx_counter}", user_id, current_date.replace(hour=14), 'Kredit',
                f"TRF MASUK - {random.choice(list_windfall)}", "Rejeki", '0000', float(win_val),
                round(sisa_saldo_user[user_id], 2), 'Pemasukan', 'Pemasukan Tambahan', 0
            ])
            id_trx_counter += 1

        curr_p = active_persona_user[user_id]
        ratio = sisa_saldo_user[user_id] / gaji
        l_needs = poisson_lambda[curr_p]['Needs']
        l_wants = poisson_lambda[curr_p]['Wants'] * peak_wants_multiplier
        l_trans = poisson_lambda[curr_p]['Transfer']

        # --- PERBAIKAN LOGIKA SURVIVAL MODE ---
        if curr_p == 'Spendthrift':
            # Spendthrift dibiarkan foya-foya tanpa mempedulikan rasio saldo (Limit dilepas)
            f_needs = np.random.poisson(l_needs)
            f_wants = np.random.poisson(l_wants)
            f_trans = np.random.poisson(l_trans)
        else:
            # Persona Unconflicted & Tightwad tetap menggunakan sistem Rem Otomatis
            f_needs = np.random.poisson(l_needs) if ratio > 0.15 else np.random.poisson(l_needs * 0.3)
            f_wants = np.random.poisson(l_wants) if ratio > 0.4 else (np.random.poisson(l_wants*0.5) if ratio > 0.15 else 0)
            f_trans = np.random.poisson(l_trans) if ratio > 0.15 else 0

        if ratio <= 0.15: survival_streak[user_id] += 1

        process_cat(user_id, 'Needs', f_needs, current_date)
        process_cat(user_id, 'Wants', f_wants, current_date)
        process_cat(user_id, 'Transfer', f_trans, current_date)

df_transaksi = pd.DataFrame(transaksi_list, columns=[
    'id_transaksi', 'id_user', 'timestamp', 'tipe_mutasi', 'deskripsi_mutasi',
    'catatan_mutasi', 'mcc', 'nominal', 'sisa_saldo', 'kategori_besar_sistem',
    'kategori_detail_sistem', 'label_anomali',
    'gt_kategori_besar', 'gt_kategori_detail'
])
df_transaksi = df_transaksi.sort_values(by=['timestamp', 'id_user']).reset_index(drop=True)
print(f"✅ BERHASIL! Total {len(df_transaksi):,} transaksi dibangitkan.")
display(df_transaksi.head())

print("\nKategori Besar Unik dalam df_transaksi:")
print(df_transaksi['kategori_besar_sistem'].unique())

🚀 TAHAP 3: Menyalakan Mesin Waktu (Enhanced with Routines & Subs)...
✅ BERHASIL! Total 55,405 transaksi dibangitkan.


,id_transaksi,id_user,timestamp,tipe_mutasi,deskripsi_mutasi,catatan_mutasi,mcc,nominal,sisa_saldo,kategori_besar_sistem,kategori_detail_sistem,label_anomali,gt_kategori_besar,gt_kategori_detail
0,TRX-10139,USR-001,2026-01-01 07:00:00,Kredit,TRF MASUK - PENDAPATAN TETAP,-,0000,3130000.0,3130000.0,Pemasukan,Pendapatan Bulanan,0,None,None
1,TRX-11261,USR-002,2026-01-01 07:00:00,Kredit,TRF MASUK - PENDAPATAN TETAP,-,0000,2200000.0,2200000.0,Pemasukan,Pendapatan Bulanan,0,None,None
2,TRX-10103,USR-003,2026-01-01 07:00:00,Kredit,TRF MASUK - PENDAPATAN TETAP,-,0000,3380000.0,3380000.0,Pemasukan,Pendapatan Bulanan,0,None,None
3,TRX-10469,USR-004,2026-01-01 07:00:00,Kredit,TRF MASUK - PENDAPATAN TETAP,-,0000,2890000.0,2890000.0,Pemasukan,Pendapatan Bulanan,0,None,None
4,TRX-11057,USR-005,2026-01-01 07:00:00,Kredit,TRF MASUK - PENDAPATAN TETAP,-,0000,1580000.0,1580000.0,Pemasukan,Pendapatan Bulanan,0,None,None



Kategori Besar Unik dalam df_transaksi:
['Pemasukan' 'Needs' 'Wants' 'Transfer']


In [ ]:
display(df_nasabah[['id_user', 'persona_dasar', 'is_dynamic', 'persona_bulan_1', 'persona_bulan_2', 'persona_bulan_3']].head(20))

,id_user,persona_dasar,is_dynamic,persona_bulan_1,persona_bulan_2,persona_bulan_3
0,USR-362,Unconflicted,0,Unconflicted,Unconflicted,Unconflicted
1,USR-074,Unconflicted,0,Unconflicted,Unconflicted,Unconflicted
2,USR-375,Tightwad,0,Tightwad,Tightwad,Tightwad
3,USR-156,Spendthrift,0,Spendthrift,Spendthrift,Spendthrift
4,USR-105,Tightwad,1,Tightwad,Tightwad,Tightwad
5,USR-395,Spendthrift,0,Spendthrift,Spendthrift,Spendthrift
6,USR-378,Unconflicted,0,Unconflicted,Unconflicted,Unconflicted
7,USR-125,Spendthrift,1,Spendthrift,Tightwad,Spendthrift
8,USR-069,Tightwad,0,Tightwad,Tightwad,Tightwad
9,USR-451,Unconflicted,0,Unconflicted,Unconflicted,Unconflicted


In [ ]:
df_transaksi.to_csv('df_transaksi.csv', index=False)
print("df_transaksi berhasil disimpan sebagai 'df_transaksi.csv'")

df_transaksi berhasil disimpan sebagai 'df_transaksi.csv'
